# Data Visualization - Lesson 3: Advanced Business Visualization

**Session length:** 6 hours  
**Theme:** Multi-dimensional analysis, deeper business diagnostics, storytelling with visualizations, critique and improvement of charts, and a larger capstone-style analysis.

This notebook uses the **Sample Superstore** retail dataset and is designed as a practical, discussion-driven class session.

## Learning goals
By the end of this session, students should be able to:
1. Analyze business performance across multiple dimensions at the same time.
2. Diagnose performance problems using charts rather than just describe them.
3. Turn analysis into a simple visual story with a beginning, middle, and recommendation.
4. Critique weak charts and redesign them more clearly.
5. Conduct a larger capstone-style business analysis using visualization and pandas.

## Teaching flow
- Part 1: Setup and review
- Part 2: Multi-dimensional analysis
- Part 3: Deeper business diagnostics
- Part 4: Storytelling with visualizations
- Part 5: Critique and improvement of charts
- Part 6: Capstone-style analysis
- Part 7: Wrap-up and extension ideas


## Instructor note

This lesson intentionally moves from:
- **basic chart reading**  
to
- **business interpretation**  
to
- **decision-oriented storytelling**

Students should be encouraged to answer questions such as:
- What is happening?
- Where is it happening?
- For whom is it happening?
- Why might it be happening?
- What should the manager do next?

That shift is the real goal of this session.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Seaborn style
sns.set_theme()


## Load the dataset

This lesson uses the **Sample Superstore** dataset, a classic retail dataset for business analysis.

The notebook first tries to load a local file named `superstore.csv`.  
If that file is not available, it falls back to a public raw CSV link.


In [ ]:
# Load data
local_path = "superstore.csv"
fallback_url = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

try:
    df = pd.read_csv(local_path, encoding="latin1")
    print("Loaded local file:", local_path)
except FileNotFoundError:
    df = pd.read_csv(fallback_url, encoding="latin1")
    print("Loaded from URL")

# Basic cleaning
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Add time helper columns
df["Order Month"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
df["Order Year"] = df["Order Date"].dt.year
df["Order Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)

df.head()


In [ ]:
# Quick structure review
print(df.shape)
print()
print(df.dtypes)


In [ ]:
# Missing values check
df.isna().sum().sort_values(ascending=False)


## Quick business context

Each row is an order line.  
A line has:
- a product
- a customer segment
- a location
- sales
- discount
- profit

This means one chart can answer a simple question, but a combination of charts can answer a management question such as:

> Which regions and categories create revenue, but destroy profit when discounts rise?


## Part 1. Warm-up review: what should an advanced analyst do?

A beginner often asks:
- Which category has the highest sales?

An advanced analyst asks:
- Which category has the highest sales **and** weak profit?
- Does that pattern vary by region?
- Is discount part of the explanation?
- Is the problem broad, or concentrated in a few sub-categories?
- Is the issue getting better or worse over time?


## Part 2. Multi-dimensional analysis

### 2.1 Category by region: sales and profit together

A single dimension is usually not enough.  
Business performance often changes when we split by both:
- **product dimension** (category / sub-category)
- **market dimension** (region / segment)


In [ ]:
category_region = (
    df.groupby(["Region", "Category"], as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"),
           orders=("Order ID", "nunique"))
)

category_region


In [ ]:
# Compare sales by region and category
plt.figure(figsize=(10, 6))
sns.barplot(data=category_region, x="Region", y="total_sales", hue="Category")
plt.title("Total Sales by Region and Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()


In [ ]:
# Compare profit by region and category
plt.figure(figsize=(10, 6))
sns.barplot(data=category_region, x="Region", y="total_profit", hue="Category")
plt.title("Total Profit by Region and Category")
plt.ylabel("Total Profit")
plt.tight_layout()
plt.show()


### Reflection questions
1. Does the category with the highest sales also have the highest profit?
2. Is any category strong in one region but weak in another?
3. If a category sells well but profits poorly, what explanations are possible?


### 2.2 Heatmap for multi-dimensional comparison

In [ ]:
profit_pivot = (
    df.pivot_table(index="Sub-Category", columns="Region", values="Profit", aggfunc="sum")
      .fillna(0)
)

plt.figure(figsize=(10, 8))
sns.heatmap(profit_pivot, annot=False, cmap="coolwarm", center=0)
plt.title("Profit Heatmap: Sub-Category by Region")
plt.tight_layout()
plt.show()


Why this chart matters:

A bar chart is good for a small number of groups.  
A **heatmap** is often better when we want to compare many combinations at once.

This is a classic multi-dimensional analysis chart because it helps students see:
- concentration
- strong areas
- weak areas
- negative profit pockets


In [ ]:
# Same idea for average discount
discount_pivot = (
    df.pivot_table(index="Sub-Category", columns="Region", values="Discount", aggfunc="mean")
      .fillna(0)
)

plt.figure(figsize=(10, 8))
sns.heatmap(discount_pivot, annot=False, cmap="YlOrRd")
plt.title("Average Discount Heatmap: Sub-Category by Region")
plt.tight_layout()
plt.show()


### Discussion
Now compare the two heatmaps:
- Do high-discount areas line up with low-profit areas?
- Are there exceptions?
- What would you investigate next?


### 2.3 Segment-level perspective

In [ ]:
segment_summary = (
    df.groupby(["Segment", "Category"], as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"))
)

segment_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=segment_summary, x="Segment", y="total_sales", hue="Category", ax=axes[0])
axes[0].set_title("Sales by Segment and Category")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=segment_summary, x="Segment", y="total_profit", hue="Category", ax=axes[1])
axes[1].set_title("Profit by Segment and Category")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()


### Mini exercise
Write 2-3 business observations from the segment analysis.

Try to go beyond:
- "Consumer has high sales."

Instead say:
- "Consumer appears to drive the most sales, but we should compare its profit efficiency because high volume does not automatically mean strong business quality."


## Part 3. Deeper business diagnostics

This section goes beyond description and moves into diagnosis.

A diagnostic question sounds like:
- Why is profit weak?
- Where is the problem concentrated?
- What patterns are associated with the problem?


### 3.1 Discount vs profit: relationship analysis

In [ ]:
sample_df = df.dropna(subset=["Discount", "Profit", "Sales"]).sample(2500, random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sample_df,
    x="Discount",
    y="Profit",
    hue="Category",
    alpha=0.6
)
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix for a few numeric variables
numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
df[numeric_cols].corr()


### Important interpretation note

Correlation helps, but it does **not** prove causation.

A manager should not say:
- "Discount causes all losses."

A better statement is:
- "Higher discounts are associated with weaker profitability in many cases, so discount policy is an important factor to investigate."


### 3.2 Find loss-making sub-categories

In [ ]:
loss_by_subcat = (
    df.groupby("Sub-Category", as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"),
           orders=("Order ID", "nunique"))
      .sort_values("total_profit")
)

loss_by_subcat.head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=loss_by_subcat.head(10),
    x="total_profit",
    y="Sub-Category"
)
plt.title("Lowest-Profit Sub-Categories")
plt.xlabel("Total Profit")
plt.tight_layout()
plt.show()


### Discussion prompt
For the weakest sub-categories:
- Are they weak because of low sales?
- Or because they sell a lot but lose money?
- Is discount high?
- Could shipping, product mix, or pricing be part of the issue?


### 3.3 Trend diagnostics over time

In [ ]:
monthly = (
    df.groupby("Order Month", as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"))
)

fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

axes[0].plot(monthly["Order Month"], monthly["total_sales"])
axes[0].set_title("Monthly Sales")
axes[0].set_ylabel("Sales")

axes[1].plot(monthly["Order Month"], monthly["total_profit"])
axes[1].set_title("Monthly Profit")
axes[1].set_ylabel("Profit")

axes[2].plot(monthly["Order Month"], monthly["avg_discount"])
axes[2].set_title("Average Discount by Month")
axes[2].set_ylabel("Discount")
axes[2].set_xlabel("Order Month")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Interpretation questions
1. Are sales and profit moving together?
2. Do we see months where sales rise but profit does not?
3. Does discount seem to rise before weak-profit periods?
4. What additional chart would strengthen your explanation?


### 3.4 A manager-facing diagnostic chart

In [ ]:
subcat_diag = (
    df.groupby("Sub-Category", as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"))
)

plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=subcat_diag,
    x="total_sales",
    y="total_profit",
    size="avg_discount",
    hue="avg_discount",
    sizes=(50, 800)
)
plt.axhline(0, linestyle="--")
plt.title("Sub-Category Diagnostic View: Sales, Profit, and Discount")
plt.tight_layout()
plt.show()


This is a more advanced business diagnostic chart because it combines:
- **x-axis:** total sales
- **y-axis:** total profit
- **size:** average discount
- **color:** average discount

This lets us ask:
- Which sub-categories have high sales but weak profit?
- Are those also the ones with larger discounts?


## Part 4. Storytelling with visualizations

Good analysis is not just a collection of charts.  
It should tell a story.

A simple business data story often has 3 stages:

### Story structure
1. **Situation**  
   What matters overall?

2. **Complication**  
   What problem or tension appears?

3. **Resolution / recommendation**  
   What should management do next?

Let's build one.


### 4.1 Story chart 1: total company trend

In [ ]:
company_year = (
    df.groupby("Order Year", as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"))
)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(company_year["Order Year"], company_year["total_sales"], marker="o")
ax1.set_title("Company Sales by Year")
ax1.set_xlabel("Year")
ax1.set_ylabel("Sales")

plt.tight_layout()
plt.show()


**Possible narration:**  
"The company is growing in sales over time. At first glance, the business appears healthy."


### 4.2 Story chart 2: weak-profit problem areas

In [ ]:
weak_subcats = loss_by_subcat.head(8)

plt.figure(figsize=(10, 6))
sns.barplot(data=weak_subcats, x="total_profit", y="Sub-Category")
plt.title("Profit Problem Areas: Lowest-Profit Sub-Categories")
plt.xlabel("Total Profit")
plt.tight_layout()
plt.show()


**Possible narration:**  
"However, profit is not healthy everywhere. A few sub-categories are dragging results down."


### 4.3 Story chart 3: likely driver

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=subcat_diag,
    x="avg_discount",
    y="total_profit",
    size="total_sales",
    hue="Category",
    sizes=(50, 700),
    alpha=0.8
)
plt.axhline(0, linestyle="--")
plt.title("Average Discount vs Total Profit by Sub-Category")
plt.xlabel("Average Discount")
plt.ylabel("Total Profit")
plt.tight_layout()
plt.show()


**Possible narration:**  
"One likely driver is discounting. Some sub-categories with higher average discount also show weak or negative profit."

### A simple recommendation slide could say:
- Review discount policy in weak sub-categories.
- Investigate pricing, vendor cost, and promotion strategy.
- Focus first on high-sales / low-profit product groups.


### Storytelling exercise

Write a short 4-6 sentence management summary using the 3 charts above.

Template:
1. Overall, the business shows ...
2. However, ...
3. The problem seems concentrated in ...
4. A likely contributing factor is ...
5. A practical next step is ...


## Part 5. Critique and improvement of charts

A strong analyst should not only make charts.  
They should also be able to critique bad ones.

This section intentionally creates a few weaker charts and then improves them.


### 5.1 Example of a cluttered chart

In [ ]:
# Intentionally crowded chart
state_profit = (
    df.groupby("State", as_index=False)
      .agg(total_profit=("Profit", "sum"))
      .sort_values("total_profit", ascending=False)
)

plt.figure(figsize=(14, 5))
plt.bar(state_profit["State"], state_profit["total_profit"])
plt.title("Profit by State")
plt.xticks(rotation=90)
plt.show()


### Critique questions
- Is this chart technically wrong?
- Is it easy to read?
- What makes it hard to use?
- What business question is it trying to answer?
- Could a filtered or sorted version be better?


In [ ]:
# Improved version: show only top and bottom performers separately
top_states = state_profit.head(10)
bottom_states = state_profit.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=top_states, x="total_profit", y="State", ax=axes[0])
axes[0].set_title("Top 10 States by Profit")

sns.barplot(data=bottom_states, x="total_profit", y="State", ax=axes[1])
axes[1].set_title("Bottom 10 States by Profit")

plt.tight_layout()
plt.show()


### Why the improved chart is better
- It matches a clearer business question.
- It reduces clutter.
- It separates top and bottom performers.
- It helps the audience notice extremes faster.


### 5.2 Pie chart critique

In [ ]:
segment_sales = (
    df.groupby("Segment", as_index=False)
      .agg(total_sales=("Sales", "sum"))
)

plt.figure(figsize=(6, 6))
plt.pie(segment_sales["total_sales"], labels=segment_sales["Segment"], autopct="%1.1f%%")
plt.title("Sales Share by Segment")
plt.show()


This pie chart is acceptable, but a bar chart often supports comparison better.


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=segment_sales, x="Segment", y="total_sales")
plt.title("Sales by Segment")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()


### Discussion
Why might the bar chart be better than the pie chart here?
- easier comparison
- easier labeling
- more precise sense of difference


### 5.3 Misleading design choices

When critiquing charts, check for:
- too many categories
- tiny fonts
- cluttered labels
- unclear titles
- decorative choices with no analytical value
- missing context
- misleading axis choices


## Part 6. Capstone-style analysis

Now students shift from guided tasks to a larger analysis.

### Capstone scenario
You are a junior analyst supporting a retail leadership team.

The VP asks:

> "Sales are growing, but profit quality seems uneven.  
> Where are our strongest and weakest areas, what seems to drive the weakness, and what actions should we prioritize?"

Your job is to use the dataset to produce:
1. a performance overview
2. a diagnosis of weak areas
3. one or two practical recommendations


### 6.1 Step 1: build an executive KPI summary

In [ ]:
kpis = pd.DataFrame({
    "metric": ["Total Sales", "Total Profit", "Total Orders", "Average Discount"],
    "value": [
        df["Sales"].sum(),
        df["Profit"].sum(),
        df["Order ID"].nunique(),
        df["Discount"].mean()
    ]
})

kpis


### 6.2 Step 2: identify strong and weak business areas

In [ ]:
capstone_summary = (
    df.groupby(["Region", "Category", "Sub-Category"], as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"),
           orders=("Order ID", "nunique"))
)

capstone_summary.head()


In [ ]:
# Top profit generators
capstone_summary.sort_values("total_profit", ascending=False).head(10)


In [ ]:
# Worst profit performers
capstone_summary.sort_values("total_profit", ascending=True).head(10)


### 6.3 Step 3: create a capstone dashboard-style set of charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: sales by category
cat_summary = df.groupby("Category", as_index=False).agg(total_sales=("Sales", "sum"))
sns.barplot(data=cat_summary, x="Category", y="total_sales", ax=axes[0, 0])
axes[0, 0].set_title("Total Sales by Category")

# Chart 2: profit by region
region_summary = df.groupby("Region", as_index=False).agg(total_profit=("Profit", "sum"))
sns.barplot(data=region_summary, x="Region", y="total_profit", ax=axes[0, 1])
axes[0, 1].set_title("Total Profit by Region")

# Chart 3: monthly profit trend
monthly_profit = df.groupby("Order Month", as_index=False).agg(total_profit=("Profit", "sum"))
axes[1, 0].plot(monthly_profit["Order Month"], monthly_profit["total_profit"])
axes[1, 0].set_title("Monthly Profit")
axes[1, 0].tick_params(axis="x", rotation=45)

# Chart 4: discount vs profit
sample_df = df.sample(2500, random_state=42)
sns.scatterplot(data=sample_df, x="Discount", y="Profit", hue="Category", alpha=0.6, ax=axes[1, 1])
axes[1, 1].set_title("Discount vs Profit")

plt.tight_layout()
plt.show()


### Capstone discussion guide
Ask students to answer:
1. What are the biggest positive performance areas?
2. What are the biggest negative performance areas?
3. Is the weakness broad or concentrated?
4. Do discounts appear to be part of the issue?
5. What would you recommend for the next management meeting?


### 6.4 Guided capstone tasks

#### Task A
Find the 5 sub-categories with the lowest total profit.  
Then visualize them.

#### Task B
Check whether the same weak sub-categories are weak in all regions or only some regions.

#### Task C
Identify one high-sales / low-profit area and explain why it deserves management attention.

#### Task D
Build a 3-chart visual story for an executive audience.

#### Task E
Write a short executive summary (150-250 words).


In [ ]:
# Your workspace starts here

# Example starter: weakest sub-categories
weakest_subcats = (
    df.groupby("Sub-Category", as_index=False)
      .agg(total_profit=("Profit", "sum"))
      .sort_values("total_profit")
      .head(5)
)

weakest_subcats


### 6.5 Extension: profit ratio analysis

In [ ]:
# Add profit ratio for more nuanced diagnosis
ratio_summary = (
    df.groupby(["Category", "Sub-Category"], as_index=False)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"))
)

ratio_summary["profit_ratio"] = ratio_summary["total_profit"] / ratio_summary["total_sales"]
ratio_summary.sort_values("profit_ratio").head(10)


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=ratio_summary.sort_values("profit_ratio").head(10),
    x="profit_ratio",
    y="Sub-Category"
)
plt.title("Lowest Profit Ratio Sub-Categories")
plt.xlabel("Profit / Sales")
plt.tight_layout()
plt.show()


This is useful because:
- total profit shows absolute weakness
- profit ratio shows efficiency weakness

A sub-category can look large and bad in absolute dollars, or small and bad in efficiency.  
Both matter, but in different ways.


## Part 7. Reflection and wrap-up

### What students should now be able to do
- Compare performance across multiple dimensions.
- Diagnose likely causes of weak performance.
- Use charts to support a management recommendation.
- Critique charts and improve them.
- Conduct a larger visual analysis project.

### Final reflection questions
1. Which chart in this notebook was most useful for diagnosis, and why?
2. Which chart was most useful for communication, and why?
3. What is the difference between a chart for exploration and a chart for presentation?
4. If you had 10 more minutes with the VP, what additional analysis would you show?


## Homework / independent project idea

Create a short business slide deck or memo that includes:
1. one overview chart  
2. one diagnostic chart  
3. one recommendation chart or table  
4. a 1-paragraph executive summary

The goal is not to make the fanciest chart.  
The goal is to help a business decision-maker act.
